# 🟫 Bronze Layer — Raw Data Ingestion (Medallion Architecture)

This notebook implements the **Bronze layer** of the Medallion Architecture for the **Magnificent 7 Stocks ETL Pipeline**.

The Bronze layer is responsible for:

- Retrieving raw stock price data directly from the source API (Yahoo Finance using `yfinance`)
- Preserving the raw format as much as possible without applying business logic
- Adding ingestion metadata such as a UTC load timestamp
- Appending all raw batches from the 7 stocks into a unified Bronze Delta table
- Ensuring reproducibility and auditability of all upstream data

The Bronze layer is designed to be **append‑only**.  
No cleanups, transformations, deduplications, or enrichments occur at this stage.  
Any data-quality fixes (such as type casting, deduplication, schema standardization, and filtering) are intentionally left for the **Silver layer**, which builds a clean, canonical dataset based on this raw Bronze history.

This ensures that Bronze remains an **immutable, fully historical record** of everything the system ingested.

## 📌 Setup & Dependencies

In [0]:
# Databricks: install yfinance (lightweight) if not already available
%pip install --quiet yfinance

# After a %pip install, Databricks recommends a restart of Python to ensure the environment is refreshed.
dbutils.library.restartPython()

In [0]:
import os, json
from pathlib import Path

import yfinance as yf
import pandas as pd

from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, StringType
from pyspark.sql import functions as F

## 📌 Load Config & Resolve Paths 

In [0]:
# ================================
# 1. Locate the repo root safely
# ================================
REPO_NAME = "magnificent7stocks-etl"   # <-- your repo folder name EXACTLY

cwd = Path(os.getcwd())
repo_root = cwd

while repo_root.name != REPO_NAME and repo_root != repo_root.parent:
    repo_root = repo_root.parent

print("───── REPO PATH VALIDATION ─────")
if repo_root.name == REPO_NAME:
    print(f"[OK] Repo root found: {repo_root}")
else:
    print(f"[ERROR] Could not locate repo '{REPO_NAME}'.")
    print("Current working dir:", cwd)
    raise FileNotFoundError("Repo root could not be found. Check REPO_NAME.")

# ================================
# 2. Validate config folder
# ================================
config_dir = repo_root / "02_config"

print("\n───── CONFIG FOLDER VALIDATION ─────")
print(f"Expected config folder: {config_dir}")

if not config_dir.exists():
    print(f"[ERROR] Config folder does NOT exist: {config_dir}")
    print("Make sure the structure is:")
    print(f"{REPO_NAME}/02_config/")
    raise FileNotFoundError("Config folder missing.")
else:
    print("[OK] Config folder found.")

# ================================
# 3. Validate individual JSON files
# ================================
tickers_path = config_dir / "tickers.json"
api_cfg_path = config_dir / "api_config.json"

print("\n───── FILE VALIDATION ─────")

# --- Tickers file ---
if tickers_path.exists():
    print(f"[OK] tickers.json found at: {tickers_path}")
    with open(tickers_path) as f:
        tickers = json.load(f).get("tickers", [])
else:
    print(f"[WARNING] tickers.json NOT found at: {tickers_path}")
    print("→ Using default tickers.")
    tickers = ["AAPL", "AMZN", "GOOGL", "META", "MSFT", "NVDA", "TSLA"]

# --- API configuration file ---
if api_cfg_path.exists():
    print(f"[OK] api_config.json found at: {api_cfg_path}")
    with open(api_cfg_path) as f:
        api_cfg = json.load(f)
else:
    print(f"[WARNING] api_config.json NOT found at: {api_cfg_path}")
    print("→ Using default API configuration.")
    api_cfg = {"source": "yfinance", "interval": "1h", "period": "1d"}

# ================================
# 4. Extract values with fallbacks
# ================================
interval = api_cfg.get("interval", "1d")
period = api_cfg.get("period", "1mo")

print("\n───── FINAL CONFIG LOADED ─────")
print("Tickers:", tickers)
print("Interval:", interval)
print("Period:", period)
print("================================\n")


## 📌 Ingest Data From Yahoo Finance  
### Why Data Extraction Is Done *Block by Block*

To ensure stable and reliable ingestion of stock market data from the Yahoo Finance API, each ticker is downloaded **individually in separate blocks** instead of using a single loop. This approach is used because:

- **Prevents API throttling**  
  Yahoo Finance often blocks or limits rapid consecutive requests from cloud environments, causing empty or incomplete responses when fetching multiple tickers in a loop.

- **Ensures complete and consistent data**  
  Downloading one ticker at a time avoids partially loaded or corrupted DataFrames that can break ETL processes.

- **Improves debugging clarity**  
  Each block shows whether a specific ticker succeeded or failed, making troubleshooting simpler and more efficient.

- **Avoids schema and concatenation issues**  
  When some tickers return empty results, merging them inside a loop can cause schema mismatches, Pandas errors, or Spark type‑casting problems. Block‑by‑block extraction eliminates this risk.

- **Follows robust ETL patterns**  
  Processing each data source independently increases reliability and prevents one failure from affecting the entire ingestion pipeline.

### 📌 Standardizing Raw API Schema

Yahoo Finance returns column names in a **MultiIndex format**, such as:

('Adj Close', 'AAPL')
('High', 'AAPL')
...

To build a consistent Bronze table across all 7 tickers, we normalize the raw schema in the boxes above by:

- **Flattening MultiIndex column names**
- **Removing the ticker suffix** (e.g., `_AAPL` → `''`)
- **Lowercasing names** and replacing spaces with underscores
- **Fixing special cases**, such as converting `datetime_` to `datetime`
- **Appending a `ticker` column** to preserve the asset identity

These steps do **not** modify the data itself — they only standardize the structure.  
This is allowed in Bronze because schema harmonization is not business transformation.


Stock 1 — AAPL

In [0]:
ticker = tickers[0]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_01 = df_raw

display(df_01)

Stock 2 — AMZN

In [0]:
ticker = tickers[1]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_02 = df_raw

display(df_02)

Stock 3 — GOOGL

In [0]:
ticker = tickers[2]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_03 = df_raw

display(df_03)

Stock 4 — META

In [0]:
ticker = tickers[3]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_04 = df_raw

display(df_04)

Stock 5 — MSFT

In [0]:
ticker = tickers[4]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_05 = df_raw

display(df_05)

Stock 6 — NVDA

In [0]:
ticker = tickers[5]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_06 = df_raw

display(df_06)

Stock 7 — TSLA

In [0]:
ticker = tickers[6]
print("Downloading:", ticker)

df_raw = yf.download(ticker, period=period, interval=interval, auto_adjust=False)
df_raw = df_raw.reset_index()

df_raw.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in df_raw.columns
]

df_raw.columns = (
    df_raw.columns
        .str.replace("_" + ticker, "", regex=False)  # remove "_TSLA"
        .str.replace(" ", "_")                      # "Adj Close" -> "Adj_Close"
        .str.lower()
)

df_raw.columns = df_raw.columns.str.replace(r"datetime_?$", "datetime", regex=True)

df_raw["ticker"] = ticker

df_07 = df_raw

display(df_07)

### 📌 Adding Ingestion Metadata (Load Timestamp)

The Bronze table includes one system-level metadata column:

- **`ingestion_ts_utc`** — the UTC timestamp indicating *when* the data was ingested.

### Why this is important:

- Enables **auditing** of ingestion events  
- Helps **debug delays or API inconsistencies**  
- Supports **incremental Silver pipelines** by filtering on ingestion time  
- Ensures **full reproducibility** of downstream layers  
- Records exactly when each batch entered the system

This timestamp is **not** a business field — it is technical metadata used for data engineering and orchestration purposes.

In [0]:
df_all = pd.concat([df_01, df_02, df_03, df_04, df_05, df_06, df_07], ignore_index=True)


# Add UTC load timestamp
df_all["ingestion_ts_utc"] = pd.Timestamp.now(tz="UTC")


display(df_all)
print("Rows:", len(df_all))
print("Columns:", list(df_all.columns))


## 📌 Convert to Spark DataFrame with schema

In [0]:
schema = StructType([
    StructField("datetime", StringType(), True),
    StructField("close_price", DoubleType(), True),
    StructField("ticker", StringType(), True)
])

# Fix: cast datetime column to string to match schema
spark_df_bronze = spark.createDataFrame(df_all)

display(spark_df_bronze)

### 📌 Write Policy: Append‑Only

The Bronze table is the **historical log** of everything ingested.  
To preserve lineage and enable complete replay of downstream layers, **we never overwrite** Bronze.  
Instead, we always **append** new batches:

- Keeps a full history of raw ingestions
- Preserves the “truth as received” from the source
- Enables reproducible rebuilds of the Silver/Gold layers

### ✳️ Operational Notes

- **Duplicates are allowed** in Bronze. De‑duplication is performed in Silver.
- **Schema drift** at ingestion time is acceptable; minimal harmonization (column naming) is applied only to ensure the table can be appended safely.
- The table is written to **Unity Catalog** so it’s discoverable, governed, and queryable across the workspace.

In [0]:
spark_df_bronze.write.format("delta").mode("append").saveAsTable("magnificent7stocks.bronze_layer")  


### 📌 Why Export CSV Files

Delta tables live in cloud storage and are not committed to GitHub.  
For transparency and easier data review, we export two lightweight CSV files from the Bronze layer:

- A **Snapshot CSV** for quick visibility into the latest ingestion  
- A **Historical CSV** containing all current Bronze data  
- Both allow inspection without needing Databricks  
- Provide simple artifacts that mirror the Bronze schema  

---

### ✳️ Conventions

#### **Snapshot CSV**
- **Folder:** `04_tables/`  
- **File name:** `bronze_layer_snapshot.csv`  
- **Content:** a snapshot of the **latest ingestion run only**  
- **Purpose:** quick preview in GitHub; not intended to store history  

#### **Historical CSV**
- **Folder:** `04_tables/`  
- **File name:** `bronze_layer_historical.csv`  
- **Content:** **all data currently stored** in the Bronze layer  
- **Purpose:** a complete dataset export (not log files), useful for offline analysis or validation  

In [0]:
df_all.to_csv("/Workspace/Users/tomasricardo1999@gmail.com/databricks-projects-showcase/magnificent7stocks-etl/04_tables/bronze_layer_snapshot.csv", index=False)

df_all_spark = spark.sql("SELECT * FROM magnificent7stocks.bronze_layer")

df_all_spark.toPandas().to_csv(
    "/Workspace/Users/tomasricardo1999@gmail.com/databricks-projects-showcase/magnificent7stocks-etl/04_tables/bronze_layer_historical.csv",
    index=False
)


### ✅ What this notebook accomplished
- Ingested raw stock prices for the **Magnificent 7** using `yfinance`
- Normalized raw column headers (flattened MultiIndex, removed ticker suffixes, standardized casing)
- Added a **UTC ingestion timestamp** (`ingestion_ts_utc`) as technical metadata
- Built a unified Spark DataFrame and **appended** it to the **Bronze** Delta table in **Unity Catalog**
- Exported two **CSVs snapshot and historical** into `04_tables/` for portfolio documentation
